# Modelagem — Score de detração pós-viagem

Este notebook treina e avalia o modelo que estima, para cada jornada encerrada, a probabilidade de o Cliente se tornar Detrator. Ele começa onde o `pre-processamento.ipynb` termina: consome a base analítica já integrada e não repete nenhuma etapa de limpeza ou integração.

Princípios adotados:

- consumir a base analítica pronta, sem reprocessar as fontes brutas;
- importar a lógica de seleção, divisão e transformação de `scripts/preprocessamento_nps.py`, nunca reescrevê-la aqui;
- ajustar imputação, codificação e escala somente no treino;
- manter o conjunto de teste intocado até a etapa de avaliação;
- versionar o notebook sem saídas de execução e sem nenhum dado do parceiro.

O notebook é preenchido por etapas, cada uma com card próprio no board:

| Seção | Conteúdo | Card |
|---|---|---|
| 1 | Contrato de entrada e carga da matriz | #100 |
| 2 | Baselines de referência | #101 |
| 3 | Validação cruzada por Cliente | #102 |
| 4 | Primeiro modelo candidato | #103 |
| 5 | Ajuste de hiperparâmetros | #104 |
| 6 | Métricas de ordenação | #105 |
| 7 | Limiar operacional e matriz de confusão | #106 |
| 8 | Curvas ROC e precisão contra cobertura | #107 |
| 9 | Tabela comparativa | #108 |


## 1. Contrato de entrada

Esta seção fixa o que a etapa de carga e organização dos dados precisa entregar para que a modelagem rode. O contrato existe para que as duas frentes avancem em paralelo: enquanto a organização não estiver pronta, basta que o arquivo abaixo exista com estas colunas para que todo o restante do notebook funcione sem alteração.

**Arquivo esperado:** `data/processed/base_analitica.parquet`, produzido pela seção 13 do `notebooks/pre-processamento.ipynb`. No Colab, o mesmo arquivo pode vir do Drive; a busca de caminho está na célula seguinte e não exige edição de código.

**Colunas obrigatórias.** A base precisa carregar três grupos:

| Grupo | Colunas | Para que serve |
|---|---|---|
| Alvo | `DETRATOR` | variável a ser predita, binária |
| Chaves de partição | `DATA_STD_CONVERTIDA`, `ID_GOLDENRECORD` | corte temporal e agrupamento por Cliente |
| Features | as 11 de `FEATURE_SET_V1` | matriz do modelo |

As 11 features são a allowlist do score pós-viagem definida no módulo de pré-processamento e não são reescritas aqui: `TIER_VIAGEM`, `VOO_TIPO`, `TIPO_ENTRETENIMENTO`, `CANAL_COMPRA`, `SEGMENTO`, `ESTATISTICA_ATRASOSAIDA`, `ATRASO_CHEGADA`, `CANCELAMENTO_VOO`, `ANTECEDENCIA_CANCELAMENTO`, `TEMPO_VOO`, `N_TRECHOS`. A lista canônica é o `FEATURE_SET_V1`, que o módulo também expõe sob o nome antigo `FEATURES_SCORE_POS_VIAGEM`, e é ele que a célula 1.1 importa. O `QTDE_VIAGENS_12M` ocupava a última posição até o `feat(#149)` e está suspenso pela seção 4.3.2.3 da documentação, enquanto não houver forma auditável de recalcular o histórico com corte em `t_score`. Qualquer mudança nessa lista é decisão de modelagem e acontece no módulo, não no notebook.

**Partições.** A divisão não é aleatória. `split.dividir`, chamada dentro de `preparar_matriz` (seção 1.4), aplica corte temporal e por Cliente em três vias — treino, validação e teste —, retirando de cada conjunto anterior todo Cliente que reaparece num conjunto mais recente, porque a base tem dependência intracliente e um Cliente presente em dois lados inflaria o desempenho estimado. A função devolve as três partições como DataFrames completos, e é pelo índice de cada uma que features, alvo e grupos são referenciados no restante do notebook, nunca por posição.

**O que não pode entrar na matriz do modelo.** Nenhuma coluna iniciada por `NPS_` ou `SUB_`. São respostas coletadas no mesmo instrumento que origina o alvo, e o contrato temporal da seção 4.2.3 da documentação as exclui do modelo.

A proibição vale para a matriz, e não para a base analítica. A base carrega essas colunas por construção: `DETRATOR` é derivado de `NPS_PRINCIPAL` dentro da própria `preparar_base_analitica`, e a persistência em Parquet da seção 13 do pré-processamento não descarta coluna alguma. Exigir uma base sem elas faria o notebook falhar sobre uma entrada correta. Por isso as duas verificações têm papéis distintos, e isso é deliberado:

| Célula | Papel | Interrompe a execução |
|---|---|---|
| 1.3 | diagnóstico da base analítica: confere colunas obrigatórias e features da allowlist, e apenas relata quantas colunas de pesquisa vieram junto | sim para coluna obrigatória ausente, não para coluna de pesquisa presente |
| 1.4 | contrato da matriz do modelo: confere que o conjunto de features é exatamente a allowlist e que nenhuma coluna de pesquisa entrou, na entrada e na saída do pré-processador | sim |


### 1.1. Dependências e módulo de pré-processamento

## Conexão com a base de dados

As bases da Azul **não são versionadas** neste repositório, por compromisso de proteção de dados firmado entre o Inteli e o parceiro: a pasta `data/` e as extensões `.csv`, `.xlsx` e `.parquet` estão no `.gitignore`. Para executar este notebook:

1. **Arquivos brutos.** Copie os arquivos `.csv` recebidos da Azul para `data/raw/`, na raiz do repositório:
   - `PROJETO_INTELI.NPS_01.csv` a `PROJETO_INTELI.NPS_04.csv`;
   - `PROJETO_INTELI.PERFIL_CLIENTE_01.csv` e `PROJETO_INTELI.PERFIL_CLIENTE_02.csv`;
   - `PROJETO_INTELI.INFORMACAO_VIAGEM.csv`;
   - `PROJETO_INTELI.DISTRIBUICAO_PAX_NORMALIZADO.csv`.

   Deixe fora de `data/raw/` as cópias `.xlsx` das mesmas bases: o pré-processamento compara os dois formatos de uma mesma parte e interrompe a execução se eles divergirem.
2. **Base analítica.** Execute `notebooks/pre-processamento.ipynb`. Ele integra os arquivos de `data/raw/` e grava `data/processed/base_analitica.parquet`, a base que os notebooks de análise e de modelagem leem.
3. **Google Colab.** Siga a seção "No Google Colab" do `README.md`: envie a pasta do projeto ao Google Drive, com as bases em `data/raw/`, e execute antes de tudo uma célula que monta o Drive e entra na pasta `notebooks/` do projeto (`%cd`). Com isso, os caminhos descritos abaixo funcionam sem ajuste, como na execução local.

**Neste notebook:** lê `data/processed/base_analitica.parquet`, gerada pelo passo 2. A seção 1.2 procura a base a partir da raiz do repositório, de `notebooks/`, de `/content/` e do Drive. Para usar outro local, acrescente o caminho à lista `CAMINHOS_POSSIVEIS`.

In [ ]:
!pip install -q pandas numpy scikit-learn pyarrow matplotlib seaborn

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Inclui a pasta do módulo tanto no projeto local quanto no Colab. O break evita
# que duas cópias de preprocessamento_nps.py fiquem no caminho ao mesmo tempo,
# o que tornaria ambíguo qual delas o import resolve.
for pasta_codigo in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import preprocessamento_nps
importlib.reload(preprocessamento_nps)

from preprocessamento_nps import (
    FEATURES_SCORE_POS_VIAGEM,
    conferir_features_v1_na_base,
    validar_parquet,
)

SEMENTE = 42
np.random.seed(SEMENTE)

# O contrato da seção 1 ancora em 11 features. Uma mudança silenciosa na
# allowlist do módulo precisa falhar aqui, e não apenas aparecer no print.
QUANTIDADE_FEATURES_ESPERADA = 11
if len(FEATURES_SCORE_POS_VIAGEM) != QUANTIDADE_FEATURES_ESPERADA:
    raise AssertionError(
        f"A allowlist mudou: {len(FEATURES_SCORE_POS_VIAGEM)} features, "
        f"esperadas {QUANTIDADE_FEATURES_ESPERADA}. "
        "Atualize o contrato de entrada da seção 1 antes de seguir."
    )

print("features do score pós-viagem:", len(FEATURES_SCORE_POS_VIAGEM))


### 1.2. Localização e carga da base analítica

A busca cobre o projeto local e o Colab, com e sem Drive montado. Se nenhum caminho existir, a célula falha com a instrução do que fazer, em vez de seguir com uma base vazia.


In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
]

caminho_base = next((caminho for caminho in CAMINHOS_POSSIVEIS if caminho.exists()), None)

if caminho_base is None:
    raise FileNotFoundError(
        "Base analítica não encontrada. Gere-a com a seção 13 de notebooks/pre-processamento.ipynb "
        "ou monte o Drive e ajuste CAMINHOS_POSSIVEIS. Caminhos testados: "
        + ", ".join(str(caminho) for caminho in CAMINHOS_POSSIVEIS)
    )

validar_parquet(caminho_base)
df = pd.read_parquet(caminho_base)

print("base analítica:", caminho_base)
print("linhas:", f"{len(df):,}".replace(",", "."), "| colunas:", df.shape[1])


### 1.3. Diagnóstico da base analítica

A célula confere o que a base precisa ter e interrompe a execução se faltar coluna obrigatória ou feature da allowlist. Falhar aqui é barato; descobrir a ausência depois de treinar não é.

A conferência da allowlist passa por `conferir_features_v1_na_base`, que antes deriva `N_TRECHOS` de `BASE_AIRPORTLEG` quando a base ainda não a traz, caso de um parquet gravado antes do #177. Só falta de verdade uma feature que não está na base nem pode ser derivada dela.

A contagem de colunas `NPS_` e `SUB_` é diagnóstico e não motivo de parada, porque a base analítica carrega essas colunas por construção, como explicado na seção 1. O que não pode acontecer é elas alcançarem a matriz do modelo, e é a seção 1.4 que bloqueia isso.


In [ ]:
COLUNAS_OBRIGATORIAS = {"DETRATOR", "DATA_STD_CONVERTIDA", "ID_GOLDENRECORD"}

faltando = COLUNAS_OBRIGATORIAS - set(df.columns)
if faltando:
    raise KeyError(f"A base não atende ao contrato de entrada. Colunas ausentes: {sorted(faltando)}")

# Deriva N_TRECHOS antes de conferir a allowlist: um parquet gravado antes do
# fix(#177) traz BASE_AIRPORTLEG, mas nao a derivada, e a conferencia direta
# sobre df.columns levantava KeyError por uma feature que o pipeline sabe gerar.
df = conferir_features_v1_na_base(df)

# Diagnóstico, e não trava: a base analítica carrega as respostas da pesquisa
# por construção, já que DETRATOR é derivado de NPS_PRINCIPAL. O bloqueio
# efetivo acontece sobre a matriz do modelo, na seção 1.4.
colunas_pesquisa = [coluna for coluna in df.columns if coluna.startswith(("NPS_", "SUB_"))]
print("colunas de pesquisa na base analítica:", len(colunas_pesquisa), "(diagnóstico, não bloqueia)")
print("prevalência de Detrator:", f"{df['DETRATOR'].mean():.2%}")


### 1.4. Matriz de modelagem e contrato da matriz

`preparar_matriz`, de `src/matriz.py`, executa numa chamada as três etapas que precisam
acontecer nesta ordem: particionar por data e por Cliente, selecionar a allowlist de features e
ajustar o pré-processador **apenas no treino**. Inverter as duas últimas seria inofensivo;
ajustar o pré-processador antes de particionar não, porque a mediana do treino passaria a
carregar informação de 2026 e a métrica de teste mediria um modelo que já viu o período que
deveria prever.

As datas de corte não são escolhidas aqui. Elas vêm do registro de decisão em
`documents/extras/politica-de-particionamento-temporal.md`, que é a fonte única: repeti-las como
constante do notebook criaria uma segunda verdade que poderia divergir da documentada.

`sem_data="treino"` envia ao treino qualquer linha sem data — hoje nenhuma na base analítica:
as cerca de 120 mil linhas de `NPS_01` que antes chegavam sem data eram efeito de um bug de
concatenação, já corrigido em `normalizar_data_std`, e hoje têm data normal. A política segue
como salvaguarda, e não hipótese descartável: se uma fonte futura voltar a chegar sem a coluna,
`verificar_anterioridade_sem_data` prova pela ordem de registro (`RESPONDENT_ID`) que essas
linhas antecedem o período datado antes de aceitá-las no treino. `split.dividir` chama essa
verificação por conta própria sempre que existir alguma linha sem data, interrompendo a
execução se a prova deixar de valer.

O contrato da matriz é cobrado nas duas pontas do pré-processador. Na entrada, pega uma allowlist
alterada; na saída, uma transformação que reintroduza uma coluna de pesquisa sob outro nome.
Qualquer uma das duas interrompe a execução.


In [ ]:
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import matriz
importlib.reload(matriz)
from matriz import preparar_matriz, resumo_da_matriz

# Datas do registro de decisão do #127, não escolhidas aqui.
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

preparo = preparar_matriz(
    df,
    corte_validacao=CORTE_VALIDACAO,
    corte_teste=CORTE_TESTE,
    sem_data="treino",
)

x_treino, x_validacao, x_teste = (preparo["x"][k] for k in ("treino", "validacao", "teste"))
y_treino, y_validacao, y_teste = (preparo["y"][k] for k in ("treino", "validacao", "teste"))
grupos_treino = preparo["grupos"]["treino"]
preprocessador = preparo["preprocessador"]
x_treino_transformado = preparo["matrizes"]["treino"]
x_validacao_transformado = preparo["matrizes"]["validacao"]
x_teste_transformado = preparo["matrizes"]["teste"]
numericas = preparo["metadados"]["numericas"]
categoricas = preparo["metadados"]["categoricas"]
metadados_divisao = preparo["metadados"]

display(resumo_da_matriz(preparo))

anterioridade = metadados_divisao["anterioridade_sem_data"]
# verificar_anterioridade_sem_data devolve dois formatos: um quando ha linha
# sem data para verificar, outro (sem as chaves de correlacao) quando o bloco
# esta vazio, que e o caso normal na base atual.
if anterioridade["linhas_sem_data"]:
    print("linhas sem data enviadas ao treino:", f"{anterioridade['linhas_sem_data']:,}".replace(",", "."))
    print("  correlação de Spearman entre ordem de registro e data:", anterioridade["correlacao_spearman"])
    print("  maior id sem data:", anterioridade["maior_id_sem_data"],
          "< menor id datado:", anterioridade["menor_id_datado"])
else:
    print("nenhuma linha sem data na base: a verificação de anterioridade não se aplica")
print("removidas por Cliente recorrente:",
      f"{metadados_divisao['linhas_removidas_por_recorrencia']:,}".replace(",", "."))
print("features:", len(metadados_divisao["features_usadas"]),
      "| numéricas:", len(numericas), "| categóricas:", len(categoricas),
      "| colunas da matriz:", metadados_divisao["colunas_da_matriz"])


### 1.5. Composição das partições e congelamento (CR05 do #127/#128, #131)

Esta seção não reparticiona: `split.resumo` recebe `preparo["particoes"]` e
`preparo["metadados"]`, já calculados na 1.4, e devolve n, intervalo de datas,
Clientes e percentual do total por partição. Chamar `split.dividir` de novo aqui
geraria a mesma divisão duas vezes no mesmo notebook, com o risco de as duas
chamadas divergirem se uma delas for editada e a outra não.

Pelo mesmo motivo, o congelamento do #131 se aplica às partições que a 1.4
produziu, e não a uma divisão própria. A divisão é temporal e por Cliente,
portanto determinística — mas determinismo não é congelamento: bastaria a base
analítica ser regerada com uma linha a mais para que os conjuntos mudassem sem
aviso, e as métricas que os cards #103 a #108 comparam entre si na Seção 4.4
deixariam de medir a mesma coisa. `congelamento.congelar_ou_conferir` grava o
artefato de índices na primeira execução e, nas seguintes, confere o hash das
partições atuais contra o registrado, interrompendo a execução se divergirem.

O artefato fica em `data/processed/particoes_modelagem.json`, coberto pelo
`.gitignore` como todo o diretório `data/`, porque índices são posições de linhas
de uma base que o Termo de Abertura não permite publicar. Junto dos índices ele
registra a semente fixada, a impressão digital da base de origem e as datas de
corte, e a leitura confere as três antes de aceitar a divisão. Trocar de base ou
de política de corte exige `regerar=True`, e não acontece por omissão.

In [ ]:
from congelamento import congelar_ou_conferir, descrever
from split import resumo

# A divisao ja aconteceu na 1.4, dentro de preparar_matriz. Aqui ela apenas e
# congelada: na primeira execucao o artefato de indices e gravado; nas seguintes
# ele e lido e o hash das particoes atuais e conferido contra o registrado.
registro_particoes = congelar_ou_conferir(
    preparo["particoes"],
    caminho_base=caminho_base,
    parametros={
        "corte_validacao": CORTE_VALIDACAO,
        "corte_teste": CORTE_TESTE,
        "sem_data": "treino",
    },
    metadados=preparo["metadados"],
)

# O notebook e versionado sem saidas de celula, entao semente, versao da base e
# hash dos indices so existem como registro se forem impressos na execucao.
print(descrever(registro_particoes))
print()

tabela_particoes = resumo(preparo["particoes"], metadados=preparo["metadados"])
print(tabela_particoes)

### 1.6. Auditoria de representatividade dos conjuntos (#130)

Antes de treinar, esta seção confere se treino, validação e teste são comparáveis entre si e
com a base completa — em taxa de detração e na distribuição de quatro variáveis-chave. Sem essa
checagem, uma queda de métrica no teste não se pode atribuir ao modelo: pode ser só uma diferença
de composição entre os períodos.

As quatro variáveis auditadas são `TIER_VIAGEM`, `VOO_TIPO`, `FAIXA_ATRASO` (a mesma taxonomia de
`src/clean.py`, reaproveitada aqui em vez de reinventada) e `SEGMENTO`. O card original citava
"região ou rota" como quarta variável, mas a base analítica não tem nenhuma coluna de região,
rota, origem ou destino. O candidato mais próximo, `VOO_INTERNACIONAL`, tem um único valor em
toda a base (`DOMESTIC`, 484.915 de 484.915 linhas) — sem variância, não audita nada — por isso
`SEGMENTO` entra no lugar.


In [ ]:
from clean import ORD_ATRASO, faixa_atraso

CATEGORICAS_AUDITORIA = {
    "TIER_VIAGEM": sorted(df["TIER_VIAGEM"].dropna().unique()),
    "VOO_TIPO": sorted(df["VOO_TIPO"].dropna().unique()),
    "SEGMENTO": sorted(df["SEGMENTO"].dropna().unique()),
}
LIMIAR_DESVIO_PP = 2.0


def perfil_particao(sub):
    faixa = faixa_atraso(sub["ESTATISTICA_ATRASOSAIDA"])
    linha = {"n": len(sub), "taxa_detracao_pct": round(sub["DETRATOR"].mean() * 100, 2)}
    for categoria in ORD_ATRASO:
        linha[f"FAIXA_ATRASO={categoria}"] = round((faixa == categoria).mean() * 100, 2)
    for coluna, valores in CATEGORICAS_AUDITORIA.items():
        for valor in valores:
            linha[f"{coluna}={valor}"] = round((sub[coluna] == valor).mean() * 100, 2)
    return linha


def contagem_particao(sub):
    """Contagem bruta por categoria, para o teste de suporte no treino.

    Um desvio em pontos percentuais pode ficar abaixo do limiar mesmo quando o
    treino tem suporte zero para uma categoria que aparece em validação ou
    teste: a proporção de uma categoria rara muda pouco entre conjuntos
    grandes, mas "pouco" e "zero" nao sao a mesma coisa para o OneHotEncoder,
    que so aprende categorias vistas no ajuste.
    """
    faixa = faixa_atraso(sub["ESTATISTICA_ATRASOSAIDA"])
    contagem = {f"FAIXA_ATRASO={c}": int((faixa == c).sum()) for c in ORD_ATRASO}
    for coluna, valores in CATEGORICAS_AUDITORIA.items():
        for valor in valores:
            contagem[f"{coluna}={valor}"] = int((sub[coluna] == valor).sum())
    return contagem


particoes_com_base = {**preparo["particoes"], "base completa": df}
tabela_auditoria = pd.DataFrame(
    {nome: perfil_particao(sub) for nome, sub in particoes_com_base.items()}
).T

tabela_auditoria["desvio_taxa_detracao_pp"] = (
    tabela_auditoria["taxa_detracao_pct"] - tabela_auditoria.loc["base completa", "taxa_detracao_pct"]
).round(2)

display(tabela_auditoria)

colunas_composicao = [
    c for c in tabela_auditoria.columns if c not in ("n", "taxa_detracao_pct", "desvio_taxa_detracao_pp")
]
desvios_composicao = tabela_auditoria.loc[
    tabela_auditoria.index != "base completa", colunas_composicao
].subtract(tabela_auditoria.loc["base completa", colunas_composicao], axis=1).round(2)

desvios_taxa = tabela_auditoria.loc[tabela_auditoria.index != "base completa", "desvio_taxa_detracao_pp"]
relevantes_taxa = desvios_taxa[desvios_taxa.abs() > LIMIAR_DESVIO_PP]
relevantes_composicao = desvios_composicao[desvios_composicao.abs() > LIMIAR_DESVIO_PP].stack().dropna()

print(f"desvios acima de {LIMIAR_DESVIO_PP} p.p. na taxa de detração:")
print("  nenhum" if relevantes_taxa.empty else relevantes_taxa.to_string())
print(f"desvios acima de {LIMIAR_DESVIO_PP} p.p. na composição das variáveis-chave:")
print("  nenhum" if relevantes_composicao.empty else relevantes_composicao.to_string())

# Suporte no treino: categoria com contagem zero no treino, mas presente em
# validação ou teste, vira vetor zero no OneHotEncoder (handle_unknown=
# "ignore", src/matriz.py) e some da informação disponível ao modelo — um
# problema de ausência de suporte, não de diferença de proporção, que o
# desvio em p.p. acima não pega.
contagens = pd.DataFrame(
    {nome: contagem_particao(sub) for nome, sub in preparo["particoes"].items()}
).T
sem_suporte_no_treino = contagens.columns[
    (contagens.loc["treino"] == 0) & (contagens.loc[["validacao", "teste"]].sum() > 0)
]
print(f"categorias sem suporte no treino, presentes em validação ou teste:")
if len(sem_suporte_no_treino) == 0:
    print("  nenhuma")
else:
    print(contagens.loc[["treino", "validacao", "teste"], sem_suporte_no_treino].to_string())

**Conclusão da auditoria (#130).** A taxa de detração é estável entre os três conjuntos —
20,43% no treino, 21,60% na validação e 20,41% no teste, contra 20,44% na base completa — com
desvio máximo de 1,16 p.p. (validação), abaixo do limiar de 2 p.p. adotado aqui.

A composição das variáveis-chave tem onze ocorrências acima do limiar, que colapsam em oito
desvios distintos: `VOO_TIPO=DIRETO` espelha `VOO_TIPO=CONEXÃO` em cada partição (mesma
magnitude, sinal oposto), então as duas contam como um único desvio de composição. Os cinco que
não envolvem `VOO_TIPO` são moderados, todos abaixo de 2,7 p.p., sem padrão monotônico ao longo
do tempo: `TIER_VIAGEM=AZUL FIDELIDADE` (+2,70 p.p.) e `TIER_VIAGEM=DIAMANTE` (-2,67 p.p.) no
treino; `FAIXA_ATRASO=a. Sem Atraso` (+2,27 p.p.) e `FAIXA_ATRASO=b. 15m - 60m` (-2,65 p.p.) na
validação; `TIER_VIAGEM=TOPAZIO` (-2,37 p.p.) no teste.

`VOO_TIPO` é a exceção relevante, e não só no teste: a participação de voos com conexão cai de
forma monotônica ao longo do tempo em todas as três partições — +2,59 p.p. no treino (32,08%
contra 29,49% na base completa), -4,28 p.p. na validação (25,21%) e -8,14 p.p. no teste (21,35%).
É deriva real entre períodos, não ruído de amostragem, e precisa ser reportada na Seção 4.3
(#116).

Duas categorias de `TIER_VIAGEM` não têm suporte algum no treino: `AZUL ONE` (189 linhas, 0,35%
do teste) e `DIAMANTE UNIQUE` (1.164 linhas, 2,18% do teste) não aparecem nem no treino nem na
validação, só no teste. Em pontos percentuais o desvio fica abaixo do limiar e passaria
despercebido, mas o problema não é de proporção: é ausência de suporte. O `OneHotEncoder`
(`handle_unknown="ignore"`, `src/matriz.py`) não interrompe a execução — transforma essas 1.353
linhas (2,5% do teste) em vetor zero para a dimensão de tier, e o modelo fica sem essa
informação justamente nas linhas mais recentes da base.

**A divisão é aceitável como está.** A taxa de detração, a variável mais próxima do que o modelo
precisa acertar, não mostra deriva relevante, e os cinco desvios de composição fora de `VOO_TIPO`
são moderados. Duas limitações reais precisam ser declaradas na Seção 4.3, não omitidas: a
deriva monotônica de `VOO_TIPO` nas três partições, e a ausência de suporte no treino para
`AZUL ONE` e `DIAMANTE UNIQUE`, que reduz a informação de tier disponível ao modelo em 2,5% do
teste.

### 1.7. Tabela e figura de composição dos conjuntos (#132)

Reaproveita `tabela_particoes`, já calculada na 1.5, para gerar a tabela de composição (n,
período coberto, taxa de detração e percentual do total) e a figura da linha do tempo do
particionamento. Os dois artefatos vão para `documents/extras/composicao-dos-conjuntos.md` e
`assets/`, prontos para a redação da Seção 4.3 no #116 — este card não escreve a prosa da seção.


In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

from graficos import PALETA_AZUL, aplicar_tema

RAIZ_REPO = caminho_base.parents[2]

# ---- tabela de composição, direto de tabela_particoes (sem digitação manual) ----
tabela_composicao = tabela_particoes[["n", "data_inicio", "data_fim", "taxa_alvo_pct", "pct_do_total"]].copy()
tabela_composicao.columns = ["n", "início", "fim", "taxa de detração (%)", "% do total"]

NOMES_EXIBICAO = {"treino": "Treino", "validacao": "Validação", "teste": "Teste"}

linhas_md = ["| Conjunto | n | Período | Taxa de detração | % do total |",
             "|---|---|---|---|---|"]
for particao, linha in tabela_composicao.iterrows():
    periodo = f"{linha['início']} a {linha['fim']}"
    n_fmt = f"{int(linha['n']):,}".replace(",", ".")
    linhas_md.append(
        f"| {NOMES_EXIBICAO[particao]} | {n_fmt} | {periodo} | "
        f"{linha['taxa de detração (%)']:.2f}% | {linha['% do total']:.1f}% |"
    )
tabela_composicao_md = "\n".join(linhas_md)
print(tabela_composicao_md)

# ---- figura da linha do tempo do particionamento ----
aplicar_tema()

data_col = pd.to_datetime(df["DATA_STD"], errors="coerce")
inicio_base, fim_base = data_col.min(), data_col.max()
corte_val, corte_teste = pd.Timestamp(CORTE_VALIDACAO), pd.Timestamp(CORTE_TESTE)

fig, ax = plt.subplots(figsize=(10, 3.6))
cores = {"treino": PALETA_AZUL[0], "validação": PALETA_AZUL[1], "teste": PALETA_AZUL[2]}
for nome, ini, fimseg in (
    ("treino", inicio_base, corte_val),
    ("validação", corte_val, corte_teste),
    ("teste", corte_teste, fim_base),
):
    ax.axvspan(ini, fimseg, color=cores[nome], alpha=0.85, label=nome.capitalize())

for corte, rotulo in ((corte_val, "corte validação"), (corte_teste, "corte teste")):
    ax.axvline(corte, color="#0A2A6B", linestyle="--", linewidth=1.2)
    ax.annotate(f"{rotulo}\n{corte.date()}", xy=(corte, 0.5), xycoords=("data", "axes fraction"),
                ha="center", va="center", fontsize=8.5, color="white", fontweight="bold",
                bbox={"boxstyle": "round,pad=0.3", "facecolor": "#0A2A6B", "edgecolor": "none", "alpha": 0.85})

ax.set_yticks([])
ax.set_xlim(inicio_base, fim_base)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%Y"))
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_title("Linha do tempo do particionamento temporal", pad=14)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.55), ncol=3, frameon=False)

notas_rodape = []
cobertura_pct = tabela_particoes["pct_do_total"].sum()
notas_rodape.append(
    f"os três conjuntos cobrem {cobertura_pct:.1f}% da base — o restante "
    f"({100 - cobertura_pct:.1f}%) foi removido por Cliente recorrente ou por não ter "
    "Cliente, sem posição temporal própria, e não aparece no eixo"
)

n_sem_data = metadados_divisao["linhas_sem_data"]
if n_sem_data:
    notas_rodape.append(
        f"{n_sem_data:,}".replace(",", ".") + " linhas sem data, enviadas ao treino (fora do eixo)"
    )

fig.text(0.02, 0.01, "\n".join(notas_rodape), fontsize=8, color="#333")

fig.subplots_adjust(bottom=0.46, top=0.88)
caminho_figura = RAIZ_REPO / "assets" / "linha-tempo-particionamento.png"
caminho_figura.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(caminho_figura)
plt.close(fig)
print("figura salva em", caminho_figura)


## 2. Baselines de referência

Um número de desempenho só significa alguma coisa quando existe um piso para compará-lo. Esta seção estabelece dois pisos, e todo modelo apresentado depois é julgado contra eles.

O primeiro é a **regra da classe majoritária**, que não aprende nada e existe para expor o quanto a acurácia engana numa base em que cerca de um em cada cinco Clientes é Detrator. O segundo é uma **regressão logística**, modelo linear e interpretável, que mede quanto do sinal já é capturado sem recorrer a interação entre variáveis. É contra esse segundo piso que a escolha de um método baseado em árvores, na seção 4, precisa se justificar.

Os dois são ajustados apenas na partição de treino e medidos sobre a mesma partição de teste que o modelo candidato usará, sem o que a comparação não seria válida.

**Escopo desta seção.** A acurácia e a precisão média aparecem aqui como piso de comparação, não como a resposta ao item (c) da Seção 4.3. A escolha e a explicação das três métricas oficiais do modelo são daquele item e ficam na seção 6 deste notebook.

### 2.1. Prevalência e função de avaliação

A prevalência da classe Detrator é o número que dá sentido a todo o resto: é ela que a precisão média de um modelo sem informação reproduz, e é dela que a acurácia do baseline trivial se aproxima por complemento.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, average_precision_score,
                             roc_auc_score)

print(f"prevalência de Detrator no treino: {y_treino.mean():.2%}")
print(f"prevalência de Detrator no teste:  {y_teste.mean():.2%}")


def avaliar_no_teste(nome, modelo):
    """Mede um modelo já treinado sobre a partição de teste temporal.

    A acurácia entra ao lado da precisão média de propósito. É a distância
    entre as duas que evidencia o efeito do desbalanceamento, e é isso que
    justifica não apresentar acurácia como medida de qualidade neste problema.

    O ROC-AUC entra porque a tabela comparativa da seção 9 cobra as três
    métricas oficiais definidas na seção 4.1.3, e ele é a única das três que
    já pode ser medida aqui: depende apenas da ordenação do score. Revocação
    em Detrator e precisão dependem do ponto de corte, que só é decidido na
    seção 7, e medi-las agora seria fixar um limiar arbitrário antes da
    seção que existe para escolhê-lo.
    """
    probabilidades = modelo.predict_proba(x_teste_transformado)[:, 1]
    return {
        "modelo": nome,
        "acuracia": accuracy_score(y_teste, modelo.predict(x_teste_transformado)),
        "precisao_media": average_precision_score(y_teste, probabilidades),
        "roc_auc": roc_auc_score(y_teste, probabilidades),
    }

### 2.2. Baseline trivial: regra da classe majoritária

O classificador responde sempre a classe mais frequente do treino, que é a de não Detrator. Ele não olha nenhuma feature.

O resultado esperado é uma acurácia próxima de quatro quintos, acompanhada de uma precisão média próxima da prevalência. É essa contradição que interessa: um modelo que erra todos os Detratores da base ainda assim exibe acurácia alta. A probabilidade que ele emite é degenerada, constante para toda linha, e é por isso que a precisão média desaba para o piso. Pela mesma razão o ROC-AUC fica em 0,5: sem variação no score não existe ordenação, e 0,5 é exatamente o valor de quem ordena ao acaso.

In [ ]:
trivial = DummyClassifier(strategy="most_frequent", random_state=SEMENTE)
trivial.fit(x_treino_transformado, y_treino)

resultado_trivial = avaliar_no_teste("classe majoritária", trivial)
print(f"acurácia:       {resultado_trivial['acuracia']:.4f}")
print(f"precisão média: {resultado_trivial['precisao_media']:.4f}")
print(f"ROC-AUC:        {resultado_trivial['roc_auc']:.4f}")

### 2.3. Baseline linear: regressão logística

O segundo piso já aprende, mas apenas efeitos aditivos: cada variável contribui com um peso fixo, sem que o modelo represente combinações entre elas.

O parâmetro de balanceamento de classe é declarado explicitamente. Sem ele a logística se ajusta à classe majoritária e passa a quase não prever Detrator, o que a tornaria indistinguível do baseline trivial e inútil como piso. A matriz chega aqui já escalonada pelo pré-processador da seção 1.4, condição para que os coeficientes sejam comparáveis entre variáveis de unidades diferentes.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistica = LogisticRegression(
    class_weight="balanced",   # declarado: sem isso o modelo colapsa na classe majoritária
    # O padrão de max_iter é 1000 e não basta aqui: o lbfgs ainda não
    # convergiu nesse limite, e um modelo interrompido antes da convergência
    # não serve como piso, porque os coeficientes ainda estariam mudando.
    # 5000 dá folga sem custo relevante nesta matriz.
    max_iter=5000,
    random_state=SEMENTE,
)
logistica.fit(x_treino_transformado, y_treino)

resultado_logistica = avaliar_no_teste("regressão logística", logistica)
print(f"acurácia:       {resultado_logistica['acuracia']:.4f}")
print(f"precisão média: {resultado_logistica['precisao_media']:.4f}")
print(f"ROC-AUC:        {resultado_logistica['roc_auc']:.4f}")

### 2.4. Coeficientes da regressão logística

Os coeficientes ficam acessíveis no notebook para leitura posterior. Eles indicam a direção com que cada variável empurra o risco de detração, e voltam a ser usados na discussão de resultados, quando for preciso dizer se o modelo candidato encontrou algo que o piso linear não via.

In [ ]:
coeficientes_logistica = pd.DataFrame({
    "feature": preprocessador.get_feature_names_out(),
    "coeficiente": logistica.coef_[0],
}).sort_values("coeficiente", key=abs, ascending=False, ignore_index=True)

print("intercepto:", float(logistica.intercept_[0]))
coeficientes_logistica.head(15)

### 2.5. Resultado consolidado dos dois pisos

A tabela abaixo é o piso de comparação que as seções seguintes consomem. A leitura que importa: o baseline trivial exibe a maior acurácia da tabela e a menor precisão média, e a regressão logística inverte essa relação. O ROC-AUC completa a leitura: o trivial fica no 0,5 de quem não ordena nada, e é a distância da logística para esse 0,5 que mede o sinal já capturado por um modelo puramente aditivo. Qualquer modelo apresentado depois precisa superar a segunda linha na precisão média para que seu ganho seja atribuível ao modelo, e não ao desbalanceamento da base.

In [ ]:
tabela_baselines = pd.DataFrame([resultado_trivial, resultado_logistica])

ganho = resultado_logistica["precisao_media"] - resultado_trivial["precisao_media"]
print(f"ganho da logística sobre o trivial na precisão média: {ganho:+.4f}")
if ganho <= 0:
    print("ATENÇÃO: a logística não superou o piso trivial; revise o balanceamento antes de seguir.")

tabela_baselines

## 3. Validação cruzada por Cliente

`GroupKFold` de 5 folds dentro do treino, agrupando por `ID_GOLDENRECORD`.

**Por que agrupada, e não aleatória.** As respostas de um mesmo Cliente não são independentes: a
hipótese 4 da seção 4.2.4 mediu que quem detratou uma vez volta a detratar com chance 4,61 vezes
maior, e que o efeito sobrevive ao controle por atraso e cancelamento. Num `KFold` comum, duas
respostas da mesma pessoa cairiam em folds diferentes, e o modelo acertaria a segunda por
reconhecer a primeira. A média dos cinco folds subiria sem que nada tivesse melhorado, e o
hiperparâmetro escolhido em cima dela seria escolhido sobre um número inflado.

É a mesma razão que governa a divisão treino/teste em `split.dividir`, aplicada um nível abaixo.
Aqui ela é ainda mais necessária: o corte temporal não protege nada dentro do treino, porque os
folds não são temporais.

**Por que folds e não um terceiro bloco temporal.** A partição já reserva os meses finais para o
teste; recortar mais um bloco encolheria o treino justamente na faixa em que o histórico de
Cliente começa a existir. Com os folds, cada linha do treino serve de validação exatamente uma
vez, sem sair do ajuste.

A conferência abaixo não é decorativa: `conferir_folds` interrompe a execução se um Cliente
aparecer na validação de dois folds, ou no ajuste e na validação do mesmo fold. O `GroupKFold`
garante as duas coisas por construção, mas a garantia depende de o vetor de grupos ser o certo, e
passar o vetor errado produz folds tecnicamente válidos e cientificamente errados.

In [ ]:
from validacao import (conferir_folds, criar_folds, metadados_da_divisao,
                       resumo_dos_folds)

# Os indices dos folds sao posicionais, entao valem igual para x_treino e para
# x_treino_transformado: preparar_matriz constroi os dois a partir das mesmas
# linhas, na mesma ordem.
folds = criar_folds(x_treino, grupos_treino)
conferir_folds(folds, grupos_treino)

print(resumo_dos_folds(folds, grupos_treino, y_treino))
print()
for chave, valor in metadados_da_divisao(preparo).items():
    print(f"{chave}: {valor}")

## 4. Primeiro modelo candidato

Classificador de gradient boosting sobre árvores, escolha justificada pela ausência de preditor dominante e pela interação entre fidelização e falha operacional documentadas na seção 4.2.1. As subseções 4.1 e 4.2 abaixo trazem os hiperparâmetros declarados, o treino do candidato e o desempenho medido na validação contra os dois pisos.


### 4.1. Hiperparâmetros declarados e treino do candidato

`HIPERPARAMETROS_CANDIDATO` declara todos os hiperparâmetros que importam, inclusive os que
coincidem com o padrão da biblioteca, porque padrão não declarado é decisão que ninguém tomou.
Dois deles merecem leitura antes de rodar:

- `early_stopping=False`. No padrão `"auto"` o `HistGradientBoostingClassifier` liga a parada
  antecipada sozinho acima de dez mil linhas e separa uma fatia **aleatória** do ajuste para
  medir. Essa fatia não respeita `ID_GOLDENRECORD`, então respostas do mesmo Cliente cairiam no
  ajuste e na medição interna, que é o vazamento que a seção 3 existe para evitar.
- `class_weight=None`, ao contrário da logística da seção 2.3. Lá a reponderação era necessária
  para o piso não colapsar na classe majoritária. Aqui as duas métricas usadas dependem apenas da
  ordenação do score, que a reponderação não melhora, e ela deslocaria a probabilidade predita
  para longe da frequência observada, o que estragaria o escore de Brier da seção 6 e o limiar
  por capacidade da seção 7.

A semente é a mesma `SEMENTE` da seção 1.1, e é ela que faz duas execuções seguidas devolverem o
mesmo número.


In [ ]:
import modelo
importlib.reload(modelo)
from modelo import (HIPERPARAMETROS_CANDIDATO, comparar_nos_folds,
                    conferir_ganho_sobre_os_pisos, criar_candidato)

for nome, valor in HIPERPARAMETROS_CANDIDATO.items():
    print(f"{nome}: {valor}")
print(f"random_state: {SEMENTE}")

# O candidato final vê a partição de treino inteira, com o pré-processador já
# ajustado na seção 1.4. É este objeto que as seções 5 a 8 consomem.
candidato = criar_candidato(semente=SEMENTE)
candidato.fit(x_treino_transformado, y_treino)

# `n_iter_` é igual a max_iter por construção, porque `early_stopping=False`
# desliga a parada antecipada. O print não mede nada que varie, serve de
# conferência da decisão: se a parada antecipada voltar a ligar sozinha, os dois
# números deixam de bater e isso aparece aqui.
print(f"\niterações de boosting: {candidato.n_iter_}"
      f" (max_iter declarado: {HIPERPARAMETROS_CANDIDATO['max_iter']})")


### 4.2. Desempenho na validação contra os dois pisos

A comparação acontece nos cinco folds da seção 3, e os três modelos passam pelos mesmos folds.
Duas decisões sustentam o número:

- **Fábrica, e não modelo pronto.** Cada fold recebe um estimador novo. Reaproveitar a instância
  já ajustada faria o segundo fold começar do ajuste do primeiro, e a média dos cinco mediria um
  modelo que já viu quase todo o treino.
- **Pré-processador reajustado dentro de cada fold.** O da seção 1.4 é ajustado na partição de
  treino inteira, o que é correto para o modelo final, mas usá-lo aqui deixaria a mediana da
  imputação e a escala carregarem informação das linhas que o fold usa como validação. O efeito
  seria pequeno e invisível, a pior combinação possível num número que decide hiperparâmetro na
  seção seguinte.

Os dois pisos são os mesmos da seção 2, com a mesma parametrização, para que a diferença medida
seja de algoritmo e não de configuração. A precisão média é a métrica da comparação porque
resume a curva de precisão contra revocação, que é a leitura adequada quando só um em cada cinco
Clientes é Detrator.

Esta é a célula mais cara do notebook: são quinze ajustes, cinco por algoritmo.


In [ ]:
from sklearn.base import clone

# As fábricas dos dois pisos clonam os objetos das células 2.2 e 2.3 em vez de
# repetir a configuração deles. `clone` devolve um estimador novo, ainda não
# ajustado, com os mesmos parâmetros, que é exatamente o que `avaliar_nos_folds`
# pede. Copiar a configuração aqui deixaria um ajuste na logística da seção 2
# fora da comparação por folds, e as duas medidas passariam a falar de modelos
# diferentes sem nada no caminho avisar.
fabricas = {
    "classe majoritária": lambda: clone(trivial),
    "regressão logística": lambda: clone(logistica),
    "gradient boosting": lambda: criar_candidato(semente=SEMENTE),
}

comparacao_validacao = comparar_nos_folds(
    fabricas, x_treino, y_treino, folds, preprocessador,
)

# CR02 do #103 como trava: sem ela, um candidato pior que o piso seguiria para o
# ajuste da seção 5 e só a tabela da seção 9 denunciaria o problema.
conferir_ganho_sobre_os_pisos(
    comparacao_validacao,
    candidato="gradient boosting",
    pisos=["classe majoritária", "regressão logística"],
)

display(comparacao_validacao)


**Leitura do resultado.** Medida na base analítica atual, a média dos cinco folds:

| Modelo | Precisão média | Desvio entre folds | ROC-AUC |
|---|---|---|---|
| classe majoritária | 0,2043 | 0,0011 | 0,5000 |
| regressão logística | 0,4766 | 0,0061 | 0,7337 |
| gradient boosting | 0,5023 | 0,0060 | 0,7424 |

O candidato supera os dois pisos, e o que sustenta a afirmação não é a média sozinha: ele fica à
frente da regressão logística **nos cinco folds, um a um**, com ganho de 0,026 na precisão média,
pouco mais de quatro vezes o desvio entre folds. Uma diferença menor que esse desvio não
autorizaria a conclusão, e é por isso que o desvio aparece na tabela.

O piso trivial se comporta como a seção 2.2 previu: precisão média de 0,2043, que reproduz a
prevalência de Detrator no treino, e ROC-AUC de 0,5, o valor de quem não ordena nada.

**O tamanho do ganho, dito com honestidade.** A distância entre o candidato e a logística é de
0,026, enquanto a distância entre a logística e o piso trivial é de 0,272. A maior parte do sinal
desta base já é linear, e a interação entre fidelização e falha operacional acrescenta uma fração
menor, consistente mas menor. Isso não desfaz a escolha do algoritmo, que segue justificada
porque o ganho é real e aparece em todos os folds, mas dimensiona o que ela entrega: a redação da
Seção 4.3 deve apresentar o boosting como um avanço medido, não como um salto. O ajuste de
hiperparâmetros da seção 5 é a próxima chance de mover esse número, e é ele que dirá se 0,026 é o
teto do método nesta base ou apenas o resultado da configuração inicial.

## 5. Ajuste de hiperparâmetros

Busca sobre taxa de aprendizado, tamanho da árvore e número de iterações, avaliada apenas nos
folds da seção 3. O conjunto de teste não participa e segue intocado até a seção 6.

O objetivo não é maximizar a métrica. É responder a pergunta que a seção 4.2 deixou aberta: os
0,026 de vantagem do candidato sobre a regressão logística eram o teto do método nesta base, ou
apenas o resultado de uma configuração inicial escolhida no papel? A resposta precisa ficar
rastreável, porque a seção 4.4 vai comparar modelos e vai precisar tirar o número daqui em vez
de reexecutar a busca.

### 5.1. Grade declarada e isolamento do conjunto de teste

`GRADE_HIPERPARAMETROS` (`src/modelo.py`) é fatorial e pequena de propósito: doze combinações
sobre três eixos. Assim cada efeito fica visível isoladamente e a tabela cabe no documento. Uma
busca aleatória maior daria talvez um número melhor, sem deixar claro qual parâmetro o produziu,
que é o oposto do que esta seção precisa entregar.

A configuração do #103 (`learning_rate` 0,05, `max_leaf_nodes` 31, `max_iter` 300) é uma das doze
células, e não um ponto medido por fora. Sem ela dentro da mesma tabela, passando pelos mesmos
folds, o ganho da busca não seria comparável: estaria sendo confrontado com um número produzido
em outra execução.

A decisão usa **precisão média**, calculada por `average_precision_score` em cada fold e depois
promediada entre os cinco folds. Essa métrica resume a curva de precisão contra revocação e é a
leitura adequada quando os Detratores são minoria; acurácia não decide a grade porque pode premiar
um modelo que nunca identifica a classe rara. O ROC-AUC é registrado como diagnóstico secundário,
mas não escolhe a configuração.

**Por que o teste não entra, e como isso se verifica.** `buscar_hiperparametros` não tem
parâmetro por onde receber a partição de teste, e os folds vêm de `validacao.criar_folds`, que
particiona apenas o treino agrupando por `ID_GOLDENRECORD`. A célula abaixo valida com `assert` a
cobertura das posições, a separação entre treino e teste e a ausência de um mesmo Cliente nos dois
lados de cada fold. A busca, duas células adiante, salva a tabela agregada em
`documents/extras/resultados/hiperparametros_candidato.json`; é JSON e não CSV porque o projeto bloqueia arquivos CSV
no Git para proteger os dados do parceiro.

In [ ]:
from modelo import (GRADE_HIPERPARAMETROS, buscar_hiperparametros,
                    combinacoes_da_grade, escolher_configuracao)

for eixo, valores in GRADE_HIPERPARAMETROS.items():
    print(f"{eixo}: {valores}")
combinacoes = combinacoes_da_grade(GRADE_HIPERPARAMETROS)
print(f"combinações: {len(combinacoes)} | ajustes: {len(combinacoes) * len(folds)}")

# Prova do CR01. Os asserts fazem a execução quebrar se a cobertura, a separação
# entre treino e teste ou o agrupamento por Cliente deixar de valer.
posicoes = np.unique(np.concatenate([np.concatenate(fold) for fold in folds]))
linhas_em_comum = x_treino.index.intersection(x_teste.index)
clientes_nos_dois_lados = max(
    len(set(grupos_treino.iloc[ajuste]) & set(grupos_treino.iloc[validacao]))
    for ajuste, validacao in folds
)
assert np.array_equal(np.sort(posicoes), np.arange(len(x_treino)))
assert linhas_em_comum.empty
assert clientes_nos_dois_lados == 0

print(f"\nposições cobertas pelos folds: {len(posicoes):,}".replace(",", ".")
      + f" de {len(x_treino):,}".replace(",", ".") + " linhas do treino")
print("linhas em comum entre treino e teste:", len(linhas_em_comum))
print("maior número de ID_GOLDENRECORD nos dois lados de um mesmo fold:",
      clientes_nos_dois_lados)

### 5.2. Busca nos folds de validação

Cada combinação passa pelos **mesmos** cinco folds, com o pré-processador reajustado dentro de
cada um, pelas razões já detalhadas na seção 4.2. Sem isso, a diferença entre duas linhas da
tabela misturaria efeito de hiperparâmetro com efeito de partição, e a escolha final seria feita
sobre essa mistura.

A tabela sai inteira, uma linha por combinação, e não apenas o vencedor: é o que permite
reconstruir a decisão sem repetir os quinze minutos de execução.

Esta é a célula mais cara do notebook: sessenta ajustes, doze combinações por cinco folds.

In [ ]:
tabela_busca = buscar_hiperparametros(
    GRADE_HIPERPARAMETROS, x_treino, y_treino, folds, preprocessador, semente=SEMENTE,
)
tabela_busca_ordenada = tabela_busca.sort_values(
    "precisao_media", ascending=False,
).reset_index(drop=True)
# O caminho sai de `caminho_base`, resolvido na seção 1.2, para que a gravação
# funcione igual rodando da raiz do projeto, de dentro de notebooks/ ou no Colab.
PASTA_ASSETS = caminho_base.parents[2] / "assets"
PASTA_ASSETS.mkdir(exist_ok=True)
PASTA_RESULTADOS = caminho_base.parents[2] / "documents" / "extras" / "resultados"
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

destino_busca = PASTA_RESULTADOS / "hiperparametros_candidato.json"
tabela_busca_ordenada.to_json(destino_busca, orient="records", indent=2)
print("tabela agregada salva em", destino_busca)
display(tabela_busca_ordenada)

**Leitura da tabela.** Medida na base analítica atual, ordenada da melhor para a pior. A tabela
abaixo arredonda para quatro casas; as doze combinações sem arredondamento ficam salvas pela
própria célula da busca em `documents/extras/resultados/hiperparametros_candidato.json`, para que a decisão possa ser
reconstruída sem repetir os quinze minutos de execução:

| `learning_rate` | `max_leaf_nodes` | `max_iter` | Precisão média | Desvio entre folds | ROC-AUC |
|---:|---:|---:|---:|---:|---:|
| 0,05 | 31 | 150 | 0,5025 | 0,0059 | 0,7426 |
| 0,05 | 31 | 300 | 0,5023 | 0,0060 | 0,7424 |
| 0,05 | 63 | 150 | 0,5021 | 0,0056 | 0,7422 |
| 0,10 | 31 | 150 | 0,5020 | 0,0060 | 0,7424 |
| 0,05 | 31 | 600 | 0,5013 | 0,0060 | 0,7416 |
| 0,10 | 31 | 300 | 0,5010 | 0,0059 | 0,7417 |
| 0,05 | 63 | 300 | 0,5009 | 0,0054 | 0,7414 |
| 0,10 | 63 | 150 | 0,5009 | 0,0055 | 0,7413 |
| 0,10 | 31 | 600 | 0,4985 | 0,0056 | 0,7399 |
| 0,05 | 63 | 600 | 0,4984 | 0,0056 | 0,7395 |
| 0,10 | 63 | 300 | 0,4981 | 0,0055 | 0,7394 |
| 0,10 | 63 | 600 | 0,4930 | 0,0053 | 0,7358 |

O resultado é consistente nos três eixos dentro da grade principal, e menor capacidade é melhor
nas comparações diretas:

- `learning_rate` 0,05 supera 0,10 nas seis comparações em que o resto é igual.
- `max_leaf_nodes` 31 supera 63 nas seis comparações em que o resto é igual.
- `max_iter` cai de 150 para 300 e para 600 nos quatro grupos de `learning_rate` e `max_leaf_nodes`.

A configuração do #103 já estava, portanto, do lado certo dos três eixos, e ligeiramente passada
do ponto apenas em `max_iter`. Não há região da grade principal em que aumentar a capacidade do
modelo melhore a validação: o boosting já esgotou o sinal que esta matriz de features oferece e o
que vem depois é ajuste ao ruído do fold de ajuste.

**O tamanho de tudo isso.** A grade inteira, da melhor à pior célula, cobre 0,0095 de precisão
média. O desvio entre folds de uma única configuração é de cerca de 0,0059. Ou seja, **a
distância entre o melhor e o pior hiperparâmetro testado é menos de duas vezes o ruído de
medição**. Para comparação, a troca de algoritmo medida na seção 4.2, da logística para o
boosting, valeu 0,026, mais de quatro vezes esse mesmo desvio. A escolha do algoritmo mudou o
resultado cerca de duas vezes e meia mais do que a grade de hiperparâmetros inteira.

### 5.3. Verificação de contorno e configuração final

O melhor ponto da grade, `max_iter` 150, é o menor valor testado nesse eixo, e vencedor na borda
é motivo de desconfiança: pode ser que o ótimo estivesse fora da grade, num valor ainda menor que
não foi medido. A célula abaixo fecha essa dúvida antes da escolha, medindo dois valores abaixo
de 150 no eixo vencedor. É uma verificação de contorno declarada, e não uma ampliação disfarçada
da grade: ela não entra na tabela da decisão.

Em seguida, `escolher_configuracao` aplica o critério. A regra é a mesma que o #103 usou para
afirmar que o candidato supera a logística: **ganho menor que o desvio entre folds não sustenta a
afirmação de que uma configuração é melhor que a outra**. Aqui ela vira critério de escolha, e
não apenas de leitura. Se o melhor da grade não superar a configuração de partida por mais que o
desvio dela, a de partida permanece, porque trocar a configuração final do modelo para perseguir
a terceira casa decimal seria escolher ruído.

In [ ]:
# Contorno no eixo vencedor, abaixo do menor valor da grade.
contorno = buscar_hiperparametros(
    {"learning_rate": [0.05], "max_leaf_nodes": [31], "max_iter": [50, 100]},
    x_treino, y_treino, folds, preprocessador, semente=SEMENTE,
)
destino_contorno = PASTA_RESULTADOS / "contorno_hiperparametros.json"
contorno.to_json(destino_contorno, orient="records", indent=2)
print("tabela de contorno salva em", destino_contorno)
display(contorno)

escolha = escolher_configuracao(tabela_busca)
for chave, valor in escolha.items():
    print(f"{chave}: {valor}")

**Conclusão do ajuste.** A verificação de contorno mostra que o ótimo está cercado, e não fora da
grade: a precisão média sobe de 0,4982 em `max_iter` 50 para 0,5020 em 100 e 0,5025 em 150, e
volta a cair em 300 e 600. O eixo tem um máximo interno, e a grade o continha. A tabela de
contorno fica salva em `documents/extras/resultados/contorno_hiperparametros.json`.

O melhor ponto medido é `learning_rate` 0,05, `max_leaf_nodes` 31 e `max_iter` 150, com precisão
média de 0,502487, contra 0,502337 da configuração do #103. **O ganho é de 0,00015, cerca de
quarenta vezes menor que o desvio de 0,0060 entre os folds daquela configuração.** Nas quatro
casas da tabela acima as duas linhas aparecem como 0,5025 e 0,5023, o que sugeriria 0,0002; o
valor sem arredondamento é o que a célula acima imprime em `ganho_sobre_a_partida` e o que fica
em `documents/extras/resultados/hiperparametros_candidato.json`. Pelo critério declarado acima, isso não é ganho: é a
mesma medida repetida em dois pontos vizinhos de um platô.

**A configuração do #103 permanece como configuração final do candidato:** `learning_rate` 0,05,
`max_leaf_nodes` 31, `max_iter` 300, com os demais hiperparâmetros como declarados em
`HIPERPARAMETROS_CANDIDATO`. Ela fica dentro do platô que vai de `max_iter` 100 a 300, e trocá-la
por 150 mudaria o modelo documentado na seção 4 sem que nada tivesse melhorado de forma
mensurável.

**O que este card entrega para a seção 4.4, dito sem inflar.** O ajuste de hiperparâmetros não
melhorou o candidato, e esse é o resultado, não uma falha da busca. Ele responde a pergunta que a
seção 4.2 deixou aberta: os 0,026 de vantagem sobre a logística são o teto do método nesta base,
e não o efeito de uma configuração inicial mal escolhida. O caminho para mover esse número não
passa por hiperparâmetro; passa por feature, que é onde o #141 e a discussão da seção 4.3 já
apontavam. Registrar isso vale mais para a comparação de modelos do que teria valido um ganho de
terceira casa decimal apresentado como melhoria.

## 6. Métricas de ordenação do score

Primeira seção em que o conjunto de teste decide alguma coisa. Até aqui ele existiu apenas como
partição reservada: os pisos da seção 2 foram medidos nele, mas nenhuma decisão de modelagem
consultou esse número, e a escolha do algoritmo e dos hiperparâmetros saiu inteira dos folds das
seções 3 e 5.

A configuração medida aqui é a que o #104 congelou, e é a mesma do #103, porque a busca não
encontrou ganho acima do ruído. Nada foi reajustado depois de olhar para o teste.

### 6.1. As três métricas, e por que estas três

O score não emite um veredito por passageiro: ele **ordena uma fila de contato pós-viagem**, que
a equipe de Experiência do Cliente percorre de cima para baixo até onde a capacidade do dia
alcança. As três métricas abaixo respondem a perguntas diferentes sobre essa fila, e nenhuma
delas substitui as outras duas.

- **Precisão média.** É a área sob a curva de precisão contra cobertura, e responde à pergunta
  operacional direta: descendo a fila a partir do topo, que proporção de quem a equipe contata é
  de fato Detrator? Ela é a métrica principal porque só um em cada cinco Clientes é Detrator, e
  nesse regime a acurácia premia quem nunca aponta a classe rara, como o piso trivial da seção
  2.2 demonstrou ao exibir a maior acurácia da tabela sendo inútil.
- **Área sob a curva ROC.** Depende apenas da ordenação e não da prevalência, então é a métrica
  que permite comparar este modelo com outro medido em outra partição, ou com o 0,5 de quem não
  ordena nada. Ela responde se o modelo separa os dois grupos, independentemente de quantos
  Detratores a janela de tempo trouxe.
- **Escore de Brier.** As duas anteriores enxergam apenas a **ordem** do score. Nenhuma das duas
  mudaria se todas as probabilidades fossem divididas por dois, e o modelo passaria a dizer
  "risco de 10%" onde o risco é de 20% sem que nada reclamasse. Isso importa aqui porque a saída
  vai para a equipe como probabilidade, e não como rótulo: a analista lê "68% de risco" e decide
  se aquilo justifica uma ligação. O Brier é a única das três que responde se esse número pode
  ser lido como risco. **Menor é melhor**, e ele mede calibração, não taxa de acerto.

`metricas_no_teste` (`src/modelo.py`) chama `predict_proba` por conta própria em vez de receber o
score pronto. Passar o rótulo binário de `predict()` no lugar da probabilidade degradaria a
precisão média e o Brier em silêncio, e o número errado pareceria apenas um modelo pior.

In [ ]:
from modelo import METRICA_PRINCIPAL, metricas_de_ordenacao, metricas_no_teste

metricas_teste = metricas_no_teste(
    {"classe majoritária": trivial,
     "regressão logística": logistica,
     "gradient boosting": candidato},
    x_teste_transformado, y_teste,
)
display(metricas_teste)

# A tabela da leitura abaixo sai deste DataFrame, e nao de digitacao: sem isso a
# secao 6.1 e a secao 9 podem divergir sobre a mesma execucao, que foi o que
# aconteceu no #108 (0,5213 contra 0,5212 para a mesma precisao media).
def _quatro_casas(valor):
    return f"{valor:.4f}".replace(".", ",")


linhas_61 = ["| Modelo | Precisão média | ROC-AUC | Brier |", "|---|---:|---:|---:|"]
for nome, linha in metricas_teste.iterrows():
    celulas = [nome, _quatro_casas(linha[METRICA_PRINCIPAL]),
               _quatro_casas(linha["roc_auc"]), _quatro_casas(linha["brier"])]
    if nome == "gradient boosting":
        celulas = [f"**{c}**" for c in celulas]
    linhas_61.append("| " + " | ".join(celulas) + " |")
print("\n" + "\n".join(linhas_61))

# Os tres ganhos citados na leitura, pela mesma fonte.
for metrica in (METRICA_PRINCIPAL, "roc_auc", "brier"):
    piso = metricas_teste.loc["regressão logística", metrica]
    alvo = metricas_teste.loc["gradient boosting", metrica]
    diferenca = piso - alvo if metrica == "brier" else alvo - piso
    print(f"ganho em {metrica} sobre a logística: {_quatro_casas(diferenca)}")

**Leitura da tabela.** Medida na partição de teste (2026-01-01 a 2026-06-30, 53.486 linhas):

| Modelo | Precisão média | ROC-AUC | Brier |
|---|---:|---:|---:|
| classe majoritária | 0,2041 | 0,5000 | 0,2041 |
| regressão logística | 0,5019 | 0,7442 | 0,1877 |
| **gradient boosting** | **0,5212** | **0,7492** | **0,1312** |

O candidato supera os dois pisos nas três métricas, que é o que o card pedia verificar. Três
leituras vão além disso:

**O piso trivial se comporta exatamente como a teoria prevê**, o que serve de conferência de que
as métricas estão sendo calculadas certo: precisão média de 0,2041, idêntica à prevalência de
Detrator no teste, e ROC-AUC de 0,5000 exato, o valor de quem não ordena nada.

**O teste não degradou em relação à validação.** O candidato tinha 0,5023 de precisão média na
média dos cinco folds e faz 0,5212 no teste. A auditoria do #130 havia registrado deriva de
`VOO_TIPO` entre os períodos e ausência de suporte no treino para `AZUL ONE` e `DIAMANTE UNIQUE`,
duas limitações reais da partição de teste. Elas não se traduziram em queda de desempenho, e é
melhor ter medido isso do que supor.

**As três métricas discordam sobre o tamanho da vantagem, e é isso que as torna necessárias.** Do
piso logístico para o candidato, o ROC-AUC sobe 0,0050, a precisão média sobe 0,0193, e o Brier
melhora 0,0564. Ou seja, a maior diferença entre os dois modelos não está na ordenação, onde eles
são parecidos, mas na qualidade da probabilidade emitida. Uma avaliação que tivesse parado nas
duas primeiras métricas concluiria que o boosting acrescenta pouco à logística.

### 6.2. Calibração: o que só o escore de Brier mostra

A célula abaixo dá a régua que falta para interpretar a coluna do Brier. O número sozinho não
diz se 0,1312 é bom: ele precisa ser comparado com o que se obtém sem modelo nenhum, emitindo
para todo mundo a mesma probabilidade constante igual à prevalência do treino. Esse preditor não
ordena nada, tem ROC-AUC de 0,5, e ainda assim é **perfeitamente calibrado na média**.

A segunda parte compara a média do score do candidato no teste com a proporção de Detratores
realmente observada. Se o modelo estiver calibrado, os dois números têm que ficar próximos.

In [ ]:
from sklearn.metrics import brier_score_loss

# Referencia de calibracao: probabilidade constante igual a prevalencia do
# treino. Nao ordena nada (ROC-AUC 0,5), mas acerta a media, e por isso serve de
# regua: um modelo com Brier pior do que este emite probabilidade pior do que
# quem nao olhou para nenhuma feature.
constante = np.full(len(y_teste), float(y_treino.mean()))
brier_constante = brier_score_loss(y_teste, constante)
print(f"Brier do preditor constante (= prevalência do treino, {y_treino.mean():.4f}):"
      f" {brier_constante:.4f}")
for nome in metricas_teste.index:
    marca = "melhor" if metricas_teste.loc[nome, "brier"] < brier_constante else "PIOR"
    print(f"  {nome}: {metricas_teste.loc[nome, 'brier']:.4f} ({marca} que a constante)")

score_candidato = candidato.predict_proba(x_teste_transformado)[:, 1]
print(f"\nscore do candidato no teste: mínimo {score_candidato.min():.4f}"
      f" | média {score_candidato.mean():.4f} | máximo {score_candidato.max():.4f}")
print(f"proporção de Detratores observada no teste: {y_teste.mean():.4f}")

**O que a calibração revela, e que a ordenação escondia.** O preditor constante igual à
prevalência tem Brier de 0,1625. Comparando a coluna do Brier com essa régua:

- **classe majoritária, 0,2041: pior que a constante.** O `DummyClassifier` com
  `strategy="most_frequent"` emite probabilidade zero para todo mundo, então afirma com certeza
  total que ninguém é Detrator, e erra em 20,41% dos casos. Não é apenas desinformado: é
  ativamente mal calibrado.
- **regressão logística, 0,1877: também pior que a constante.** Este é o resultado que mais
  importa registrar. A logística ordena bem, com ROC-AUC de 0,7442, e ainda assim emite
  probabilidade pior do que quem chuta a média. A causa está declarada na própria seção 2.3: ela
  usa `class_weight="balanced"`, necessário ali para o piso não colapsar na classe majoritária,
  e a reponderação empurra a probabilidade predita para longe da frequência observada. O
  comentário de `HIPERPARAMETROS_CANDIDATO` (`src/modelo.py`) previu exatamente isso ao justificar
  `class_weight=None` no candidato, e esta seção é onde a previsão se confirma.
- **gradient boosting, 0,1312: melhor que a constante**, e é o único dos três que é.

A segunda conferência fecha o argumento: a média do score do candidato no teste é 0,1976, contra
0,2041 de Detratores observados. O modelo subestima o risco médio em 0,0065, menos de um ponto
percentual, e o score cobre a faixa de 0,0394 a 0,9646, ou seja, discrimina de fato em vez de
concentrar todo mundo perto da média.

**Conclusão para a seção 7 e para a redação da 4.3.** A probabilidade que este modelo emite pode
ser lida como risco, o que é a pergunta da User Story deste card. Isso não é consequência
automática de o modelo ordenar bem, e a linha da regressão logística nesta mesma tabela é a prova
disso: dois modelos com ROC-AUC praticamente igual, um deles utilizável como risco e o outro não.
É essa propriedade que permite à seção 7 escolher o limiar operacional em cima da probabilidade,
e não apenas em cima do lugar na fila.

## 7. Limiar operacional e matriz de confusão

A seção 6 mostrou que a probabilidade emitida pode ser lida como risco. Esta seção transforma
essa probabilidade em ação, escolhendo o ponto de corte que separa quem entra na fila de contato
de quem fica fora.

**O limiar não é uma decisão de modelagem, é uma decisão de operação.** Ele não sai de maximizar
uma métrica agregada: sai da capacidade de contato da equipe de Experiência do Cliente. Uma lista
maior do que a operação consegue percorrer descreve um processo que não existe, e o número que
acompanha essa lista não significa nada para quem vai trabalhá-la.

Por isso `0,5` não é candidato aqui. Esse corte só faria sentido se contatar quem não detrataria
custasse o mesmo que deixar passar um Detrator, e a seção 4.1.3 registra que a própria Azul
considera o segundo erro mais caro.

### 7.1. A premissa de capacidade, declarada como premissa

**`CAPACIDADE_DIARIA` abaixo é suposição do grupo, e não número fornecido pela Azul.** Nenhum
documento do parceiro define quantos contatos pós-viagem a equipe consegue fazer por dia, então o
valor entra aqui como parâmetro explícito, para ser substituído assim que a companhia confirmar o
número real. A seção 7.2 existe justamente para que essa substituição não obrigue a refazer a
análise.

O período de teste cobre 181 dias corridos, todos com resposta, a uma média de 295,5 respostas
por dia. Uma capacidade de 50 contatos diários significa trabalhar cerca de 17% do fluxo, que é a
ordem de grandeza de "algumas centenas por dia" discutida com a equipe. O tamanho da fila no
período é `k = 50 x 181 = 9.050`, e o limiar é o score da 9.050-ésima resposta mais arriscada.

In [ ]:
from modelo import (limiar_por_capacidade, matriz_de_confusao, metricas_no_topo,
                    tabela_de_capacidade)

# PREMISSA DO GRUPO, nao dado do parceiro. Trocar por um numero confirmado pela
# Azul refaz toda a secao sem nenhuma outra alteracao.
CAPACIDADE_DIARIA = 50
DIAS_NO_TESTE = 181

k_operacional = CAPACIDADE_DIARIA * DIAS_NO_TESTE
score_teste = candidato.predict_proba(x_teste_transformado)[:, 1]

print(f"Detratores no teste: {int(y_teste.sum()):,}".replace(",", ".")
      + f" de {len(y_teste):,}".replace(",", ".") + f" ({y_teste.mean():.2%})")
print(f"respostas por dia no período: {len(y_teste) / DIAS_NO_TESTE:.1f}")

operacional = metricas_no_topo(y_teste, score_teste, k_operacional)
for chave, valor in operacional.items():
    print(f"{chave}: {valor}")

### 7.2. Sensibilidade da premissa

Uma única linha de resultado esconderia o que mais importa aqui: a premissa de capacidade é o
parâmetro mais incerto de toda a seção, e a leitura muda bastante com ele. A tabela abaixo
calcula o limiar, a precisão no topo e a cobertura para várias capacidades, de modo que a
confirmação do número real pela Azul selecione uma linha já calculada em vez de exigir uma nova
análise.

A faixa vai até 250 contatos por dia porque acima disso a fila passa a alcançar quase todo o
fluxo diário de 295,5 respostas, e deixa de ser uma fila priorizada.

In [ ]:
capacidades = tabela_de_capacidade(
    y_teste, score_teste, [25, 50, 100, 150, 200, 250], dias=DIAS_NO_TESTE,
)
display(capacidades[["n_selecionados", "limiar", "precisao_no_topo", "cobertura", "f1"]])

# As metas da secao 4.1.3: revocacao >= 0,70 e precisao >= 0,40 na classe
# Detrator. A varredura abaixo procura um tamanho de fila que atenda as duas ao
# mesmo tempo, em vez de afirmar que atende ou que nao atende.
import numpy as np

ordem = np.argsort(score_teste)[::-1]
y_ordenado = np.asarray(y_teste)[ordem]
acumulado = np.cumsum(y_ordenado)
tamanhos = np.arange(1, len(y_ordenado) + 1)
cobertura_k = acumulado / int(y_teste.sum())
precisao_k = acumulado / tamanhos

atende_as_duas = (cobertura_k >= 0.70) & (precisao_k >= 0.40)
print("existe fila que atende revocação >= 0,70 e precisão >= 0,40 ao mesmo tempo:",
      bool(atende_as_duas.any()))
k_70 = int(np.argmax(cobertura_k >= 0.70) + 1)
print(f"  revocação 0,70 exige fila de {k_70:,}".replace(",", ".")
      + f" ({k_70 / DIAS_NO_TESTE:.0f} contatos/dia), e a precisão lá é {precisao_k[k_70 - 1]:.4f}")
k_40 = int(np.argmax(precisao_k < 0.40) + 1)
print(f"  precisão cai abaixo de 0,40 a partir de {k_40:,}".replace(",", ".")
      + f" ({k_40 / DIAS_NO_TESTE:.0f} contatos/dia), onde a cobertura é {cobertura_k[k_40 - 1]:.4f}")

**Leitura da sensibilidade.** Medido no conjunto de teste, com 10.919 Detratores em 53.486
respostas:

| Contatos/dia | Fila no período | Limiar | Precisão no topo | Cobertura | F1 |
|---:|---:|---:|---:|---:|---:|
| 25 | 4.525 | 0,5188 | 0,7052 | 0,2922 | 0,4132 |
| **50** | **9.050** | **0,2934** | **0,5386** | **0,4464** | **0,4882** |
| 100 | 18.101 | 0,1673 | 0,3856 | 0,6393 | 0,4810 |
| 150 | 27.150 | 0,1283 | 0,3078 | 0,7655 | 0,4391 |
| 200 | 36.211 | 0,1006 | 0,2603 | 0,8633 | 0,4000 |
| 250 | 45.301 | 0,0824 | 0,2267 | 0,9406 | 0,3654 |

A troca é a esperada e é ela que torna a escolha uma decisão de operação: filas curtas acertam
muito e alcançam pouco, filas longas alcançam quase todos e desperdiçam a maior parte das
ligações. Na premissa de 50 contatos por dia, a equipe liga para 9.050 pessoas no semestre e
4.874 delas são Detratoras, ou seja, **53,86% de acerto contra os 20,41% que uma lista aleatória
daria**, um ganho de 2,6 vezes sobre o acaso, alcançando 44,64% de todos os Detratores do período.

O F1 tem seu valor máximo justamente em 50 contatos por dia (0,4882), mas isso é coincidência e
não critério: a capacidade foi fixada antes de olhar para a tabela, e escolher o limiar pelo pico
de uma métrica agregada é exatamente o erro que produz uma lista que a operação não consegue
trabalhar.

**As duas metas da seção 4.1.3 não são atingíveis ao mesmo tempo por este modelo, e isso precisa
ser declarado.** A seção 4.1.3 define revocação mínima de 0,70 na classe Detrator e precisão
mínima de 0,40. A varredura mostra que a revocação de 0,70 exige uma fila de 22.141 respostas, ou
122 contatos por dia, e nesse ponto a precisão é de 0,3452, abaixo da meta. No sentido inverso, a
precisão cai abaixo de 0,40 a partir de uma fila de 16.921, ou 93 contatos por dia, e a
cobertura nesse ponto é de 0,6198, abaixo da meta. Não existe tamanho de fila que satisfaça as duas condições
simultaneamente. Isso não invalida o modelo: significa que o par de metas foi definido antes de
existir medição, e que uma das duas precisa ser renegociada com a Azul, ou o modelo precisa de
features novas. A seção 4.4 é o lugar de registrar essa decisão.

### 7.3. Matriz de confusão no limiar escolhido

Os quatro quadrantes abaixo estão rotulados pela ação da operação, e não por 0 e 1, porque é isso
que eles significam: a linha é o que aconteceu de fato com o Cliente, a coluna é o que a equipe
fez com ele.

In [ ]:
confusao = matriz_de_confusao(y_teste, score_teste, operacional["limiar"])
display(confusao)

print(f"limiar: {operacional['limiar']:.4f}"
      f" | fila: {operacional['n_selecionados']:,}".replace(",", ".")
      + f" | precisão no topo: {operacional['precisao_no_topo']:.4f}"
      f" | cobertura: {operacional['cobertura']:.4f}")
print(f"ganho sobre uma lista aleatória: "
      f"{operacional['precisao_no_topo'] / y_teste.mean():.2f}x")

**Os quatro quadrantes em linguagem de operação**, no limiar de 0,2934 e na premissa de 50
contatos por dia:

- **4.874 Detratores na fila (verdadeiros positivos).** São os contatos que a operação existe
  para fazer: Clientes que de fato responderam como Detratores e que a equipe alcançou antes de o
  relacionamento se deteriorar.
- **4.176 não Detratores na fila (falsos positivos).** Ligações gastas com quem não detrataria de
  qualquer forma. O custo é de alguns minutos de analista por caso, e o contato em si não é
  hostil: é uma conversa de pós-viagem com um Cliente que estava satisfeito.
- **6.045 Detratores fora da fila (falsos negativos).** O erro caro. São Clientes que detrataram
  e que ninguém procurou, e cada um representa um relacionamento perdido sem que a companhia
  tenha tido a chance de agir.
- **38.391 não Detratores fora da fila (verdadeiros negativos).** A maior parte da base, que
  corretamente não consome tempo da equipe.

**A assimetria de custo, que nenhuma métrica captura sozinha.** A seção 4.1.3 registra, com base
na conversa com a equipe de Experiência do Cliente da Azul, que deixar passar um Detrator é mais
caro do que acionar um Cliente que já era Promotor: o primeiro implica perda de relacionamento, o
segundo apenas gasto sem retorno. Essa assimetria empurraria o limiar para baixo, ampliando a
fila para reduzir os 6.045 falsos negativos. O que impede isso não é o modelo, é a capacidade: a
operação só percorre o que cabe no dia, e ampliar a fila sem ampliar a equipe apenas transfere os
falsos negativos para o fim de uma lista que ninguém alcança.

**É por isso que a conversa sobre o limiar é, na verdade, uma conversa sobre dimensionamento.**
A tabela da seção 7.2 dá o preço de cada nível de cobertura em contatos por dia, e essa é a
informação que a Azul precisa para decidir, não o limiar em si.

## 8. Curvas ROC e de precisão contra cobertura

As seções 6 e 7 entregaram números em pontos fixos: três métricas agregadas e um limiar. Esta
seção mostra o que acontece **entre** esses pontos, que é o que a operação precisa ver antes de
fixar a capacidade. Cada ponto de cada curva é um limiar possível, e a forma da curva é o custo
de mudar de ideia sobre o tamanho da fila.

As duas figuras trazem a linha do que um sorteio daria, para o ganho do modelo ser lido contra um
piso e não em termos absolutos, e marcam o limiar operacional da seção 7.1 no mesmo ponto, porque
recebem o limiar e derivam as coordenadas em vez de recebê-las prontas.

**A ROC vem primeiro por convenção, não por importância.** Numa base com 20% de prevalência ela é
a mais otimista das duas: mede se o modelo ordena bem, e ordenar bem é compatível com uma fila de
contato pouco densa. A figura que sustenta a decisão de operação é a segunda.

In [ ]:
import graficos
importlib.reload(graficos)
from graficos import g10_curva_roc, g11_precisao_cobertura

# O limiar vem da secao 7.1, e nao de um valor digitado aqui: as duas figuras
# marcam o mesmo ponto porque derivam as coordenadas dele.
figuras = {
    "g10_curva_roc": g10_curva_roc(y_teste, score_teste, operacional["limiar"]),
    "g11_precisao_cobertura": g11_precisao_cobertura(
        y_teste, score_teste, operacional["limiar"]),
}

for nome, figura in figuras.items():
    destino = PASTA_ASSETS / f"{nome}.png"
    figura.savefig(destino, dpi=130, bbox_inches="tight")
    print("figura salva em", destino)

display(*figuras.values())

**Leitura das duas curvas.** Medidas na partição de teste, com 10.919 Detratores em 53.486
respostas, e com o limiar operacional de 0,2934 marcado nas duas.

**Atenção ao numerar as figuras no documento.** Os blocos abaixo estão prontos para colar em
`documents/documentacao.md`, mas a numeração sequencial do documento reage a qual card mergeia
por último, e não à ordem em que as branches foram abertas. Pela ordem de leitura estas duas
entram na Seção 4.3, logo depois da Figura 9 (linha do tempo do particionamento) e antes do
Anexo, então são a **Figura 10** e a **Figura 11**. Só que na `develop` de hoje esses dois
números pertencem a outras figuras: `Figura 10 – Distribuição de TEMPO_VOO` e
`Figura 11 – Distribuição de ATRASO_CHEGADA`, no Anexo A.1.1. **Escrever estas duas na Seção 4.3
obriga a subir as três do anexo de 10, 11 e 12 para 12, 13 e 14 no mesmo commit**, senão o
documento passa a ter duas Figuras 10. Conferir contra a `develop` no momento em que a prosa for
escrita, porque o conflito muda se outra figura entrar antes.

<div align="center">
  <sub>Figura 10 – Curva ROC do modelo candidato</sub><br>
  <img src="../assets/g10_curva_roc.png" width="70%" alt="Curva ROC do modelo candidato sobre a partição de teste, com a diagonal do classificador aleatório e o ponto do limiar operacional marcado"><br>
  <sup>Fonte: Autoria própria.</sup>
</div>

<div align="center">
  <sub>Figura 11 – Precisão contra cobertura do modelo candidato</sub><br>
  <img src="../assets/g11_precisao_cobertura.png" width="70%" alt="Curva de precisão contra cobertura sobre a partição de teste, com a linha da prevalência como piso do sorteio e o ponto do limiar operacional marcado"><br>
  <sup>Fonte: Autoria própria.</sup>
</div>

**O que a ROC mostra, e o que ela esconde.** A curva se afasta da diagonal em toda a extensão, o
que confirma a ordenação já medida na seção 6. Mas o eixo horizontal dela é a taxa de falsos
positivos sobre os não Detratores, que são 79,6% da base: acionar milhares de pessoas erradas
move esse eixo muito pouco, porque o denominador é enorme. É por isso que uma ROC bonita convive
com uma fila de contato em que quase metade das ligações se perde, e é por isso que ela não pode
ser o argumento principal aqui.

**O que a curva de precisão contra cobertura mostra.** O eixo vertical é a pergunta da operação:
de cada cem ligações, quantas encontram um Detrator. A linha tracejada é o que uma fila sorteada
ao acaso entregaria, e a distância entre a curva e ela é o ganho real do modelo. A queda é
monotônica e acentuada: o topo da fila é denso, e cada ampliação da capacidade compra cobertura
pagando em precisão.

**O ponto marcado é uma escolha de operação, não um ótimo da curva.** Ele está onde está porque a
premissa de 50 contatos por dia o colocou ali, e não porque alguma métrica tenha sido maximizada.
Mover o ponto para a direita na segunda figura é exatamente a decisão que a seção 7.2 tabelou: mais
Detratores alcançados, menos acerto por ligação, e a mesma equipe.

## 9. Tabela comparativa

As seções anteriores mediram cada coisa no seu lugar: a ordenação na seção 6, o limiar na 7, as
curvas na 8. Esta seção junta tudo numa tabela só, com uma linha por modelo, porque é assim que a
seção 4.4 do documento precisa receber e porque separado permite o erro que o card #108 existe
para impedir, que é apresentar o número do candidato sem escala de comparação ao lado.

**Duas armadilhas de comparação, e como esta tabela evita cada uma.**

A primeira é comparar métrica dependente de corte entre modelos medidos em limiares diferentes. A
tabela resolve fixando a **capacidade**, e não o valor numérico do limiar: os três modelos
recebem a mesma fila de `k = 9.050` contatos, e cada um escolhe o próprio corte para preenchê-la.
Igualar o número seria a comparação errada, porque 0,29 na escala do candidato não significa a
mesma coisa que 0,29 na escala da logística.

A segunda é misturar resultado de validação com resultado de teste na mesma coluna. Toda a tabela
sai da partição de teste, e nenhum número dela vem dos folds das seções 3 e 5.

In [ ]:
from modelo import (METRICA_PRINCIPAL, ganhos_do_candidato,
                    tabela_comparativa)

comparativo = tabela_comparativa(
    {"classe majoritária": trivial,
     "regressão logística": logistica,
     "gradient boosting": candidato},
    x_teste_transformado, y_teste, k=k_operacional,
)
display(comparativo)

# CR04: o ganho em valor absoluto e em razao, nas duas metricas que decidem.
# So o par permite julgar relevancia: 0,02 sozinho nao diz se e muito, e a
# razao sozinha inflaria qualquer diferenca sobre um piso proximo de zero.
METRICAS_DO_GANHO = (METRICA_PRINCIPAL, "roc_auc", "precisao_no_topo")
PISOS = ["classe majoritária", "regressão logística"]
ganhos = {
    metrica: ganhos_do_candidato(comparativo, "gradient boosting", PISOS, metrica=metrica)
    for metrica in METRICAS_DO_GANHO
}
for metrica, tabela in ganhos.items():
    print(f"\nganho do candidato em {metrica}:")
    display(tabela)

comparativo.to_json(PASTA_RESULTADOS / "comparativo_modelos.json", orient="index", indent=2)
print("\ntabela salva em", PASTA_RESULTADOS / "comparativo_modelos.json")


# ---- markdown gerado do MESMO DataFrame, e nao digitado ----------------------
# Sem esta parte a tabela do documento vira copia manual: trocar o candidato
# atualizaria o JSON e deixaria o markdown com os numeros da execucao anterior.
def _v(valor, casas=4):
    return f"{valor:.{casas}f}".replace(".", ",")


def _milhar(valor):
    return f"{int(valor):,}".replace(",", ".")


CABECALHO = ("| Modelo | Precisão média | ROC-AUC | Brier | Limiar | Fila | "
             "Fila comparável | Precisão no topo | Cobertura | F1 |")
linhas_md = [CABECALHO, "|---|---:|---:|---:|---:|---:|:-:|---:|---:|---:|"]
for nome, linha in comparativo.iterrows():
    celulas = [
        nome, _v(linha[METRICA_PRINCIPAL]), _v(linha["roc_auc"]), _v(linha["brier"]),
        _v(linha["limiar"]), _milhar(linha["n_na_fila"]),
        "sim" if linha["fila_comparavel"] else "não",
        _v(linha["precisao_no_topo"]), _v(linha["cobertura"]), _v(linha["f1"]),
    ]
    if nome == "gradient boosting":
        celulas = [f"**{c}**" for c in celulas]
    linhas_md.append("| " + " | ".join(celulas) + " |")

ROTULO = {METRICA_PRINCIPAL: "Precisão média", "roc_auc": "ROC-AUC",
          "precisao_no_topo": "Precisão no topo"}
linhas_ganho = ["| Métrica | Piso | Valor do piso | Candidato | Ganho absoluto | Razão |",
                "|---|---|---:|---:|---:|---:|"]
for metrica, tabela in ganhos.items():
    for piso, linha in tabela.iterrows():
        linhas_ganho.append(
            f"| {ROTULO[metrica]} | {piso} | {_v(linha['valor_do_piso'])} | "
            f"{_v(linha['valor_do_candidato'])} | +{_v(linha['ganho_absoluto'])} | "
            f"{_v(linha['razao'], 2)}x |"
        )

bloco = ("## Tabela comparativa\n\n" + "\n".join(linhas_md)
         + "\n\n## Ganho do candidato sobre cada piso\n\n" + "\n".join(linhas_ganho))
print("\n" + bloco)

# Substitui apenas o trecho entre os marcadores: a prosa do arquivo continua
# escrita a mao, so as tabelas sao regeneradas.
INICIO, FIM = "<!-- TABELAS-GERADAS:inicio -->", "<!-- TABELAS-GERADAS:fim -->"
destino_md = RAIZ_REPO / "documents" / "extras" / "comparativo-modelos.md"
texto = destino_md.read_text(encoding="utf-8")
antes, _, resto = texto.partition(INICIO)
_, _, depois = resto.partition(FIM)
destino_md.write_text(f"{antes}{INICIO}\n\n{bloco}\n\n{FIM}{depois}", encoding="utf-8")
print("\ntabelas regeneradas em", destino_md)

**Leitura da tabela comparativa.** Partição de teste, 53.486 respostas com 10.919 Detratores
(20,41%), pedindo a todos os modelos o mesmo orçamento de 9.050 contatos. O que segue é uma
fotografia da saída da célula acima; a versão canônica, gerada do mesmo DataFrame, fica em
`documents/extras/comparativo-modelos.md`, e os valores sem arredondamento em
`documents/extras/resultados/comparativo_modelos.json`.

| Modelo | Precisão média | ROC-AUC | Brier | Limiar | Fila | Fila comparável | Precisão no topo | Cobertura | F1 |
|---|---:|---:|---:|---:|---:|:-:|---:|---:|---:|
| classe majoritária | 0,2041 | 0,5000 | 0,2041 | 0,0000 | 53.486 | não | 0,2041 | 1,0000 | 0,3391 |
| regressão logística | 0,5019 | 0,7442 | 0,1877 | 0,5993 | 9.050 | sim | 0,5283 | 0,4379 | 0,4788 |
| **gradient boosting** | **0,5212** | **0,7492** | **0,1312** | **0,2934** | **9.050** | **sim** | **0,5386** | **0,4464** | **0,4882** |

**O mesmo `k` foi pedido aos três, e só dois conseguiram cumpri-lo.** Pedir o mesmo orçamento não
garante capacidade efetiva igual, e é por isso que a igualdade não pode ser afirmada a partir do
parâmetro: ela se lê na coluna `Fila comparável`. O piso trivial emite a mesma probabilidade para
toda linha, o limiar cai sobre essa constante e a seleção alcança a partição inteira, 53.486 e não
9.050. **As cinco métricas dependentes de corte da linha do trivial descrevem uma fila de outro
tamanho e não são comparáveis às das outras duas neste orçamento.** Ele permanece na tabela porque
as três primeiras métricas, que não dependem de corte, seguem comparáveis, e porque a própria
incapacidade de formar fila é informação sobre o piso.

Não há desempate embutido para cortar a fila degenerada em exatamente 9.050. Qualquer regra desse
tipo teria de escolher quais empatados entram, e sobre um score constante essa escolha é
arbitrária: o resultado passaria a depender da ordem das linhas, e não do modelo. Um número
comparável por construção e sem significado seria pior do que a coluna que declara a limitação.

**O ganho sobre o piso trivial é grande e não é a notícia.** São 2,55 vezes em precisão média.
Superar um modelo que não olha nenhuma feature é o mínimo que se exige, e é para isso que o piso
linear existe ao lado.

**O ganho sobre a regressão logística é pequeno, e é aqui que a leitura precisa ser honesta.** Em
precisão média são 0,0193 (razão de 1,04); na precisão da fila, 0,0103 (razão de 1,02); em
ROC-AUC, 0,0050 (razão de 1,01). Traduzido para a operação: na mesma fila de 9.050 contatos, o
candidato encontra 4.874 Detratores contra 4.781 da logística, ou seja, **93 Detratores a mais no
semestre**. É ganho real e consistente nas três métricas, mas é da ordem de meio Detrator por dia
de operação, e apresentá-lo como salto seria distorcer.

**A maior diferença entre os dois está no Brier, e o que ela sustenta tem limite.** O escore cai
de 0,1877 para 0,1312, e o candidato é o único dos três melhor do que o preditor constante igual à
prevalência (0,1625, seção 6.2). Isso sustenta que ele emite a melhor probabilidade dos três pelo
erro quadrático e que a média do score fica próxima da frequência observada, 0,1976 contra 0,2041.

O que isso **não** sustenta é que a probabilidade possa ser lida como risco faixa a faixa. O Brier
agrega calibração e discriminação num número só, e uma melhora dele pode vir de qualquer um dos
dois componentes; a conferência da seção 6.2 é de média global, e média próxima não demonstra
calibração por faixa. A curva de calibração prevista na seção 4.1.3 é o que fecharia essa
afirmação, e ela ainda não existe no projeto. Até lá, o que a tabela autoriza dizer é que o
candidato tem o melhor erro probabilístico dos três e acerta a média, não que cada faixa do score
corresponda à taxa observada naquela faixa.

**Uma meta da seção 4.1.3 não foi atingida, por margem.** O ROC-AUC ficou em 0,7492 contra a meta
de 0,75. A diferença é de 0,0008 e não muda nenhuma conclusão prática, mas arredondar para 0,75 no
documento seria afirmar que a meta foi cumprida quando não foi. A precisão média, meta de 0,40,
ficou em 0,5212, com folga.